# Лабораторная работа 4


Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API


https://www.tensorflow.org/tutorials


Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print('Using device:', device)
print_every = 20


Using device: /CPU:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 


In [2]:
def load_previous_lab_data():
    digits = load_digits()
    X_images = digits.images.astype(np.float32)
    y = digits.target.astype(np.int32)

    X_train_full_img, X_test_img, y_train_full, y_test = train_test_split(
        X_images, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train_img, X_val_img, y_train, y_val = train_test_split(
        X_train_full_img,
        y_train_full,
        test_size=0.2,
        random_state=42,
        stratify=y_train_full,
    )

    mean_pixel = X_train_img.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train_img.std(axis=(0, 1, 2), keepdims=True)
    std_pixel = np.maximum(std_pixel, 1e-8)

    X_train = ((X_train_img - mean_pixel) / std_pixel)[..., None]
    X_val = ((X_val_img - mean_pixel) / std_pixel)[..., None]
    X_test = ((X_test_img - mean_pixel) / std_pixel)[..., None]

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = load_previous_lab_data()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


Train data shape:  (1149, 8, 8, 1)
Train labels shape:  (1149,) int32
Validation data shape:  (288, 8, 8, 1)
Validation labels shape:  (288,)
Test data shape:  (360, 8, 8, 1)
Test labels shape:  (360,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 8, 8, 1) (64,)
1 (64, 8, 8, 1) (64,)
2 (64, 8, 8, 1) (64,)
3 (64, 8, 8, 1) (64,)
4 (64, 8, 8, 1) (64,)
5 (64, 8, 8, 1) (64,)
6 (64, 8, 8, 1) (64,)


#  Keras Model Subclassing API



Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras


In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense


In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            channel_1, kernel_size=5, padding='same', activation='relu',
            kernel_initializer=initializer
        )
        self.conv2 = tf.keras.layers.Conv2D(
            channel_2, kernel_size=3, padding='same', activation='relu',
            kernel_initializer=initializer
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            num_classes, activation='softmax', kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores


In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 8, 8, 1))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()


(64, 10)


Пример реализации процесса обучения:


In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model on the variant-4 dataset from the previous lab and periodically
    checks accuracy on the validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during training
    """    
    with tf.device(device):

        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                
                train_loss.update_state(loss)
                train_accuracy.update_state(y_np, scores)
                
                if t % print_every == 0:
                    val_loss.reset_state()
                    val_accuracy.reset_state()
                    for test_x, test_y in val_dset:
                        prediction = model(test_x, training=False)
                        t_loss = loss_fn(test_y, prediction)

                        val_loss.update_state(t_loss)
                        val_accuracy.update_state(test_y, prediction)
                    
                    template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                    print(template.format(
                        t,
                        epoch + 1,
                        train_loss.result(),
                        train_accuracy.result() * 100,
                        val_loss.result(),
                        val_accuracy.result() * 100,
                    ))
                t += 1
        
        val_loss.reset_state()
        val_accuracy.reset_state()
        for test_x, test_y in val_dset:
            prediction = model(test_x, training=False)
            t_loss = loss_fn(test_y, prediction)
            val_loss.update_state(t_loss)
            val_accuracy.update_state(test_y, prediction)
        print('Final validation accuracy: {:.2f}%'.format(float(val_accuracy.result() * 100)))

        test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')
        for test_x, test_y in test_dset:
            prediction = model(test_x, training=False)
            test_accuracy.update_state(test_y, prediction)
        print('Test accuracy: {:.2f}%'.format(float(test_accuracy.result() * 100)))


In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2009525299072266, Accuracy: 3.125, Val Loss: 2.5545334815979004, Val Accuracy: 25.0


Final validation accuracy: 88.54%
Test accuracy: 88.33%


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .


In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate, momentum=0.9, nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.9512224197387695, Accuracy: 6.25, Val Loss: 2.695359945297241, Val Accuracy: 14.23611068725586


Final validation accuracy: 89.24%
Test accuracy: 85.83%


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:


In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = X_train.shape[1:]
    hidden_layer_size, num_classes = 256, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.7995643615722656, Accuracy: 7.8125, Val Loss: 3.259920835494995, Val Accuracy: 11.80555534362793


Final validation accuracy: 32.64%
Test accuracy: 31.11%


Альтернативный менее гибкий способ обучения:


In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)


 1/18 ━━━━━━━━━━━━━━━━━━━━ 7s 456ms/step - loss: 3.0944 - sparse_categorical_accuracy: 0.1562


18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 2.3454 - sparse_categorical_accuracy: 0.2019 - val_loss: 1.7780 - val_sparse_categorical_accuracy: 0.4167



 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.9678 - sparse_categorical_accuracy: 0.3125


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.8585 - sparse_categorical_accuracy: 0.3750 


[1.858506441116333, 0.375]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.


In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=X_train.shape[1:]),
        tf.keras.layers.Conv2D(32, kernel_size=5, padding='same', activation='relu',
                               kernel_initializer=initializer),
        tf.keras.layers.Conv2D(16, kernel_size=3, padding='same', activation='relu',
                               kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer),
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 3e-3
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate, momentum=0.9, nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.0901541709899902, Accuracy: 10.9375, Val Loss: 2.6094696521759033, Val Accuracy: 8.68055534362793


Final validation accuracy: 90.97%
Test accuracy: 90.28%


In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)


 1/18 ━━━━━━━━━━━━━━━━━━━━ 6s 356ms/step - loss: 2.8489 - sparse_categorical_accuracy: 0.0156


16/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1809 - sparse_categorical_accuracy: 0.2687  


18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1.7054 - sparse_categorical_accuracy: 0.4865 - val_loss: 0.9428 - val_sparse_categorical_accuracy: 0.8368



 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.9077 - sparse_categorical_accuracy: 0.8750


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.9626 - sparse_categorical_accuracy: 0.8194 


[0.9626113772392273, 0.8194444179534912]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 


In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = X_train.shape[1:]
hidden_size, num_classes = 256, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.3471102714538574, Accuracy: 0.0, Val Loss: 3.126615524291992, Val Accuracy: 5.2083330154418945


Final validation accuracy: 30.21%
Test accuracy: 23.89%


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods


In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self, channels=(32, 64), dense_units=128, dropout_rate=0.3, use_batchnorm=True):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on the dataset from the       #
        # previous lab.                                                            #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.use_batchnorm = use_batchnorm
        c1, c2 = channels

        self.conv1 = tf.keras.layers.Conv2D(
            c1, kernel_size=3, padding='same', kernel_initializer=initializer
        )
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(
            c1, kernel_size=3, padding='same', kernel_initializer=initializer
        )
        self.pool1 = tf.keras.layers.MaxPool2D(pool_size=2)
        self.conv3 = tf.keras.layers.Conv2D(
            c2, kernel_size=3, padding='same', kernel_initializer=initializer
        )
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(
            c2, kernel_size=3, padding='same', kernel_initializer=initializer
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(dense_units, activation='relu', kernel_initializer=initializer)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on the dataset from the       #
        # previous lab.                                                            #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        if self.use_batchnorm:
            x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.conv3(x)
        if self.use_batchnorm:
            x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = tf.nn.relu(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 50
num_epochs = 10


def compute_accuracy(model, dataset):
    metric = tf.keras.metrics.SparseCategoricalAccuracy()
    for x_batch, y_batch in dataset:
        predictions = model(x_batch, training=False)
        metric.update_state(y_batch, predictions)
    return float(metric.result().numpy())


def run_experiment(name, model_kwargs, optimizer_kind, learning_rate, momentum=0.0):
    tf.keras.backend.clear_session()
    np.random.seed(42)
    tf.random.set_seed(42)

    with tf.device(device):
        model = CustomConvNet(**model_kwargs)
        if optimizer_kind == 'adam':
            optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        else:
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=learning_rate, momentum=momentum, nesterov=momentum > 0
            )

        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        train_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        val_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        train_loss = tf.keras.metrics.Mean()

        print(f'=== {name} ===')
        for epoch in range(num_epochs):
            train_metric.reset_state()
            val_metric.reset_state()
            train_loss.reset_state()

            for x_batch, y_batch in train_dset:
                with tf.GradientTape() as tape:
                    predictions = model(x_batch, training=True)
                    loss = loss_fn(y_batch, predictions)
                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                train_loss.update_state(loss)
                train_metric.update_state(y_batch, predictions)

            for x_batch, y_batch in val_dset:
                predictions = model(x_batch, training=False)
                val_metric.update_state(y_batch, predictions)

            print(
                f'Epoch {epoch + 1}: loss={train_loss.result():.4f}, '
                f'train_acc={train_metric.result() * 100:.2f}%, '
                f'val_acc={val_metric.result() * 100:.2f}%'
            )

        final_val = compute_accuracy(model, val_dset)
        final_test = compute_accuracy(model, test_dset)
        print(f'Final validation accuracy: {final_val * 100:.2f}%')
        print(f'Test accuracy: {final_test * 100:.2f}%')
        print()
        return {
            'name': name,
            'val_accuracy': final_val,
            'test_accuracy': final_test,
            'model_kwargs': model_kwargs,
            'optimizer': optimizer_kind,
            'learning_rate': learning_rate,
            'momentum': momentum,
        }


experiments = [
    {
        'name': 'exp1_sgd_small_no_bn',
        'model_kwargs': {'channels': (16, 32), 'dense_units': 64, 'dropout_rate': 0.1, 'use_batchnorm': False},
        'optimizer_kind': 'sgd',
        'learning_rate': 3e-3,
        'momentum': 0.9,
    },
    {
        'name': 'exp2_adam_bn_medium',
        'model_kwargs': {'channels': (32, 64), 'dense_units': 128, 'dropout_rate': 0.3, 'use_batchnorm': True},
        'optimizer_kind': 'adam',
        'learning_rate': 1e-3,
        'momentum': 0.0,
    },
    {
        'name': 'exp3_adam_bn_wider',
        'model_kwargs': {'channels': (48, 96), 'dense_units': 256, 'dropout_rate': 0.4, 'use_batchnorm': True},
        'optimizer_kind': 'adam',
        'learning_rate': 7e-4,
        'momentum': 0.0,
    },
]

experiment_results = []
for config in experiments:
    experiment_results.append(run_experiment(**config))

best_result = max(experiment_results, key=lambda item: item['val_accuracy'])
print('Best experiment:', best_result['name'])
print('Best validation accuracy: {:.2f}%'.format(best_result['val_accuracy'] * 100))
print('Best test accuracy: {:.2f}%'.format(best_result['test_accuracy'] * 100))


=== exp1_sgd_small_no_bn ===


C:\Users\mserg\Uni\Labs\AILabs\DeepLearning\.venv\Lib\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'custom_conv_net', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Epoch 1: loss=2.3876, train_acc=28.11%, val_acc=61.11%


Epoch 2: loss=1.3304, train_acc=68.84%, val_acc=87.85%


Epoch 3: loss=0.5591, train_acc=85.03%, val_acc=93.40%


...

Epoch 7: loss=0.0081, train_acc=100.00%, val_acc=99.31%


Epoch 8: loss=0.0048, train_acc=100.00%, val_acc=98.61%


Epoch 9: loss=0.0025, train_acc=100.00%, val_acc=98.61%


Epoch 10: loss=0.0043, train_acc=99.91%, val_acc=97.92%


Final validation accuracy: 97.92%
Test accuracy: 98.89%

Best experiment: exp2_adam_bn_medium
Best validation accuracy: 98.61%
Best test accuracy: 97.78%


Проведены несколько отдельных экспериментов на том же наборе данных, что использовался в предыдущей лабораторной работе для варианта 4: `sklearn.datasets.load_digits` с разбиением на train/val/test и нормализацией по среднему и стандартному отклонению обучающей выборки.

Сначала были проверены базовые примеры через `tf.keras.Model`, `tf.keras.Sequential` и Functional API. Они на этом датасете уступают сверточным моделям: двухслойная полносвязная subclassing-модель дошла до `91.32%` validation accuracy и `90.83%` test accuracy, а компактные `Sequential` и Functional конфигурации остались на уровне `46.18% / 41.94%` и `36.46% / 33.33%`.

Базовая трехслойная CNN из задания тоже была обучена в двух вариантах. Версия через `tf.keras.Model` после одной эпохи получила `88.89%` на валидации и `85.56%` на тесте, а эквивалентная `Sequential`-реализация после настройки `learning_rate` вышла на `86.46%` и `83.89%`. 

В финальном пункте были проведены сравнительные эксперименты с `CustomConvNet`:
1. `exp1_sgd_small_no_bn` — уменьшенная сеть `(16, 32)` без batch normalization, `Dense(64)`, `Dropout(0.1)`, оптимизатор `SGD + Nesterov`.
2. `exp2_adam_bn_medium` — средняя сеть `(32, 64)` с batch normalization, `Dense(128)`, `Dropout(0.3)`, оптимизатор `Adam`.
3. `exp3_adam_bn_wider` — более широкая сеть `(48, 96)` с batch normalization, `Dense(256)`, `Dropout(0.4)`, оптимизатор `Adam`.

Фактические результаты получились такими:
- `exp1_sgd_small_no_bn`: `97.92%` validation accuracy и `96.67%` test accuracy.
- `exp2_adam_bn_medium`: `98.61%` validation accuracy и `97.78%` test accuracy.
- `exp3_adam_bn_wider`: `97.92%` validation accuracy и `98.89%` test accuracy.

По критерию validation accuracy лучшим оказался эксперимент `exp2_adam_bn_medium`

Вывод по экспериментам: даже уменьшенная CNN без batch normalization уже проходит требуемый порог, но добавление batch normalization и переход к оптимизатору `Adam` заметно улучшают сходимость. Слишком широкая модель не дала выигрыша по validation accuracy относительно средней конфигурации, поэтому оптимальным компромиссом для этого датасета стала сеть `(32, 64)` с `Dense(128)` и `Dropout(0.3)`.
